# Bronze Layer — Orders
## SalesFlow Data Lakehouse | Phase 2: Raw Ingestion

Ingests `orders.csv` from the Bronze volume into the `salesflow_dev.bronze.orders` Delta table.  
Data is stored as-is from the source, with control metadata added. No transformations applied.

| | |
|---|---|
| **Source** | `/Volumes/salesflow_dev/bronze/sales_data/orders.csv` |
| **Destination** | `salesflow_dev.bronze.orders` |
| **Format** | Delta Lake |
| **Load mode** | Overwrite |

In [0]:
-- Set the active catalog and schema to avoid fully qualified names throughout the notebook
USE CATALOG salesflow_dev;
USE SCHEMA bronze

In [0]:
-- Confirm we are operating in the correct catalog and schema before proceeding
SELECT current_catalog(), current_schema()

## 1. Explore Data Source
Preview the raw CSV file directly from the volume before ingestion.

In [0]:
-- Quick raw preview using direct CSV path reference (no options applied)
SELECT *
FROM csv.`/Volumes/salesflow_dev/bronze/sales_data/orders.csv`

## 2. Reset Target Table
Drops the existing table if present to ensure a clean overwrite on first load.

In [0]:
-- Drop existing table to ensure a clean state before ingestion
-- Safe to run: table will be recreated in the next step
DROP TABLE IF EXISTS salesflow_dev.bronze.orders;

## 3. Ingestion Pipeline
Reads the CSV, adds control metadata columns, saves as Delta, and validates the result.

In [0]:

%python
from pyspark.sql.functions import current_timestamp, lit
# -------------------------------------------------------
# 1. Read CSV from Bronze volume
# - sep=";" matches the SQL Server export format
# - inferSchema automatically detects column data types
# -------------------------------------------------------
df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("sep", ";") \
    .option("inferSchema", "true") \
    .load("/Volumes/salesflow_dev/bronze/sales_data/orders.csv")
)

# -------------------------------------------------------
# 2. Add control metadata columns
# - ingestion_timestamp: when this record was loaded
# - source_system: identifies the origin system
# - file_name: traceability back to the source file
# -------------------------------------------------------
df = (
    df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("SQL_Server_OnPremise"))
    .withColumn("file_name", lit("orders.csv"))
)

# -------------------------------------------------------
# 3. Save as Delta Table
# - mode overwrite: full reload on each run (first load)
# -------------------------------------------------------
df.write.mode("overwrite").format("delta").saveAsTable("salesflow_dev.bronze.orders")

# -------------------------------------------------------
# 4. Validations
# -------------------------------------------------------
orders_table = spark.table("salesflow_dev.bronze.orders")

print(f"Total records: {orders_table.count()}")
print("\nSchema:")
orders_table.printSchema()
print("\nFirst 5 rows:")
display(orders_table.limit(5))